## Configure local inputs

Set the following environment variables to your files or folders before launching Jupyter: `MOUSEGPT_KEYPOINTS_JSON`, `MOUSEGPT_OUTPUT_DIR`. Paths may include spaces. Output variables select destinations; input variables select existing session data. Review the analysis parameters in the cells before running on a new experiment.


In [ ]:
import os
required_paths = ['MOUSEGPT_KEYPOINTS_JSON', 'MOUSEGPT_OUTPUT_DIR']
missing_paths = [name for name in required_paths if not os.environ.get(name)]
if missing_paths:
    raise ValueError("Set these environment variables before running: " + ", ".join(missing_paths))


In [ ]:
import json
import numpy as np
from tqdm import tqdm
import cv2

from scipy.signal import savgol_filter

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.gridspec import GridSpec

In [ ]:
import os
print('Loading keypoints json ……')

json_path = os.environ['MOUSEGPT_KEYPOINTS_JSON']
with open(json_path, 'r') as f:
    data = json.load(f)
    
frame_num = len(data)
print(f'total {frame_num} frames')

In [ ]:
ears_pos = []

for frame in tqdm(data, desc='loading kpts'):
    ears = np.array(frame['instances'][0]['keypoints'][0:2])
    ears_pos.append(ears)

ears_pos = np.array(ears_pos)
print(ears_pos.shape)

In [ ]:
ear_speed = []

for index in tqdm(range(1, frame_num), desc='cal speed'):
    speed = np.linalg.norm(ears_pos[index]-ears_pos[index-1], axis=1)
    ear_speed.append(speed)

ear_speed = np.array(ear_speed)

# min speed of ear
ear_speed = np.concatenate([ear_speed, np.min(ear_speed, axis=1).reshape(-1,1)], axis=1)

print(ear_speed.shape)
print(ear_speed[:5])

In [ ]:
def filter(speed):
    # Apply the Savitzky-Golay filter to the 3D keypoint data
    window_length = 40
    poly_order = 7

    # Ensure window length is odd
    if window_length % 2 == 0:
        window_length += 1

    filtered_data_3d = np.empty_like(speed)

    # Apply the filter for each dimension separately
    for ear in range(speed.shape[1]):
            filtered_data_3d[:, ear] = savgol_filter(speed[:, ear], window_length, poly_order)
    return filtered_data_3d

filtered_ear_speed = filter(ear_speed)

print(filtered_ear_speed[:5])

In [ ]:
from scipy.signal import find_peaks

peak_height = 5

peaks, infos = find_peaks(filtered_ear_speed[:, 2], height=peak_height)

drop = []

for i in range(1, len(peaks)):
    if peaks[i] - peaks[i-1] <= 240:
        drop.append(i)
        drop.append(i-1)
peaks = [peaks[i] for i in range(len(peaks)) if i not in drop]

print(len(peaks))


In [ ]:
import os

out_folder = os.environ['MOUSEGPT_OUTPUT_DIR']
os.makedirs(f'{out_folder}/vis_peak_C2M2', exist_ok=True)

cap = cv2.VideoCapture(f'{out_folder}/C2M2_10mgkg_DiPT_batch.mp4')
output_fps = cap.get(cv2.CAP_PROP_FPS)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

for peak in tqdm(peaks):
    out = cv2.VideoWriter(f'{out_folder}/vis_peak_C2M2/{peak}.mp4', fourcc, output_fps, (1280, 800))
    
    cap.set(cv2.CAP_PROP_POS_FRAMES, peak-120)
    
    for i in range(480):
        ret, frame = cap.read()
        
        out.write(frame)
        
    out.release()
    
    